In [1]:
# gpt4o_extraction_test_openrouter_baseline_order.py
# Uses OpenRouter
# Uses OPENROUTER_API_KEY from system environment
# Uses temperature = 0
# Keeps ORIGINAL option order (A, B, C)
# Uses two-stage extraction:
#   1) regex extraction
#   2) LLM-based fallback extraction
# Saves raw_output and extraction_method
# Output filename includes extraction method

import json
import pandas as pd
import os
import time
import re
from openai import OpenAI, RateLimitError

# === GPT-4o configuration via OpenRouter ===
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

gpt_model = "openai/gpt-4o"
extractor_model = "openai/gpt-4o"

TEMPERATURE = 0
EXTRACTION_NAME = "regex_llm"

# Instructions (kept exactly the same)
instructions = (
    "You will be shown a scenario with a placeholder [ ]. "
    "The scenario includes three answer options labeled A, B, and C. "
    "Options may be shown in any order; ALWAYS choose by letter (A/B/C), not position. "
    "Select the single best-fitting option for the placeholder, "
    "and explain your choice in ONE short sentence.\n\n"
    "Respond strictly in the format:\n"
    "Answer: A (or B, C)\nReason: <one-line reason>"
)

def is_valid_choice(choice: str) -> bool:
    return isinstance(choice, str) and choice.upper() in {"A", "B", "C"}

def is_valid_answer(ans: str) -> bool:
    if not isinstance(ans, str):
        return False
    s = ans.strip()
    if s in ("", "Error", "Failed after 3 retries", "Invalid"):
        return False
    s = re.sub(r'^\s*Answer:\s*', '', s, flags=re.IGNORECASE).strip()
    first = re.split(r'\s|:|-|\)|\(', s)[0].upper()
    return first in {"A", "B", "C"}

def extract_reason(raw_output: str) -> str:
    if not isinstance(raw_output, str) or not raw_output.strip():
        return ""
    m_reason = re.search(r"Reason:\s*(.*)", raw_output, flags=re.IGNORECASE | re.DOTALL)
    if m_reason:
        return m_reason.group(1).strip()
    return ""

def regex_extract_choice(raw_output: str):
    if not isinstance(raw_output, str) or not raw_output.strip():
        return None

    content = raw_output.strip()

    if "Answer:" in content:
        parts = content.split("Reason:")
        answer = parts[0].replace("Answer:", "").strip()
        answer = re.sub(r'^\s*Answer:\s*', '', answer, flags=re.IGNORECASE).strip()
        first = re.split(r'\s|:|-|\)|\(', answer)[0].upper()
        if first in {"A", "B", "C"}:
            return first

    m = re.search(r'\b([ABC])\b', content[:150], flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()

    return None

def llm_extract_choice(prompt: str, raw_output: str):
    extractor_system = (
        "You are an answer extraction system.\n"
        "Your task is to identify which single option (A, B, or C) the model selected.\n"
        "You will be given the original prompt and the model's raw response.\n"
        "Return ONLY one uppercase letter: A, B, or C.\n"
        "If the answer is impossible to determine, return ONLY NONE."
    )

    extractor_user = (
        f"Original prompt:\n{prompt}\n\n"
        f"Model response:\n{raw_output}\n\n"
        "Which option did the model choose? Return only A, B, C, or NONE."
    )

    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=extractor_model,
                messages=[
                    {"role": "system", "content": extractor_system},
                    {"role": "user", "content": extractor_user}
                ],
                temperature=0
            )
            content = (resp.choices[0].message.content or "").strip().upper()
            m = re.search(r'\b([ABC]|NONE)\b', content)
            if not m:
                return None

            choice = m.group(1)
            if choice in {"A", "B", "C"}:
                return choice
            return None

        except RateLimitError:
            print("Extractor rate limit hit. Retrying in 5s...")
            time.sleep(5)
        except Exception as e:
            print(f"Extractor error: {e}")
            time.sleep(3)

    return None

def build_answer_string(choice: str, raw_output: str) -> str:
    reason = extract_reason(raw_output)
    return f"{choice} - {reason}" if reason else choice

def call_gpt4(prompt: str, instructions: str):
    """
    Returns:
        raw_output, extracted_choice, extraction_method
    """
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=gpt_model,
                messages=[
                    {"role": "system", "content": instructions},
                    {"role": "user", "content": prompt}
                ],
                temperature=TEMPERATURE
            )

            raw_output = (resp.choices[0].message.content or "").strip()

            choice = regex_extract_choice(raw_output)
            if is_valid_choice(choice):
                return raw_output, choice, "regex"

            choice = llm_extract_choice(prompt, raw_output)
            if is_valid_choice(choice):
                return raw_output, choice, "llm"

            return raw_output, None, "failed"

        except RateLimitError:
            print("Rate limit hit. Retrying in 5s...")
            time.sleep(5)
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(3)

    return "Failed after 3 retries", None, "failed"

# ORIGINAL baseline order (A, B, C)
def format_options(options_dict: dict):
    order = ["A", "B", "C"]
    lines = []
    available = [k for k in order if k in options_dict]
    for k in available:
        lines.append(f"{k}: {options_dict[k]}")
    return "\n".join(lines), ",".join(available)

# === MAIN PIPELINE ===
with open("Autonomy_Dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

csv_file = f"{TEMPERATURE}-GPT4o-ABC-{EXTRACTION_NAME}.csv"

# Resume support: keep only valid rows
if os.path.exists(csv_file):
    existing_df = pd.read_csv(csv_file)

    if "answer" not in existing_df.columns:
        existing_df["answer"] = ""

    mask_valid = existing_df["answer"].apply(lambda a: is_valid_answer(str(a)))
    if (~mask_valid).any():
        print(f"Cleaning {(~mask_valid).sum()} invalid rows from previous run...")
        existing_df = existing_df[mask_valid].reset_index(drop=True)
        tmp = csv_file + ".tmp"
        existing_df.to_csv(tmp, index=False)
        os.replace(tmp, csv_file)

    done_ids = set(existing_df["scenario_code"].tolist())
    results = existing_df.to_dict("records")
    print(f"Resuming... {len(done_ids)} scenarios already completed.")
else:
    done_ids = set()
    results = []

# Iterate scenarios
for domain in data:
    for scenario in domain["scenarios"]:
        sc_id = scenario["scenario_code"]
        if sc_id in done_ids:
            continue

        options_block, presented_order = format_options(scenario["options"])
        prompt = f"{scenario['text']}\n\n{options_block}"

        print(f"\n--- {sc_id} ---")
        print(prompt)

        raw_output, choice, extraction_method = call_gpt4(prompt, instructions)

        if not is_valid_choice(choice):
            print(f"Invalid/failed extraction for {sc_id}: {raw_output!r}. Not saving; will retry next run.")
            continue

        answer = build_answer_string(choice, raw_output)

        results.append({
            "scenario_code": sc_id,
            "answer": answer,
            "presented_order": presented_order,
            "raw_output": raw_output,
            "extraction_method": extraction_method,
            "temperature": TEMPERATURE,
            "model": gpt_model
        })

        df = pd.DataFrame(results)
        tmp = csv_file + ".tmp"
        df.to_csv(tmp, index=False)
        os.replace(tmp, csv_file)

        print(f"Raw Output: {raw_output}")
        print(f"Extracted Choice: {choice} ({extraction_method})")

print(f"\n✅ All valid results saved to {csv_file}")


--- EDU01_EPI01 ---
You are preparing your thesis. Your advisor recommends a standard method. You identify recent research that supports a different approach. You decide to …

A: implement the advisor’s recommended method.
B: adopt the alternative approach supported by recent research.
C: integrate both methods in your work.
Raw Output: Answer: C  
Reason: Integrating both methods allows you to respect your advisor's recommendation while also incorporating innovative research.
Extracted Choice: C (regex)

--- EDU01_EPI02 ---
You are advising a student on their thesis. You recommend a standard method. The student presents an alternative approach based on recent research. You decide to …

A: ask the student to use the recommended method.
B: accept the student’s proposed approach.
C: integrate both methods in their work.
Raw Output: Answer: B
Reason: Accepting the student's proposed approach encourages innovation and acknowledges recent research advancements.
Extracted Choice: B (regex)
